# Step 10d-2: trọng số tĩnh chọn trên train so với QuDAR, trên 8 bộ test và 5 bộ train

Mở rộng phát hiện 2 (Step 6) từ 2 lên 5 bộ có train (FiQA, HotpotQA, SciFact, NFCorpus, FEVER) và trả lời câu hỏi RQ3 "cần bao nhiêu nhãn". Script `scripts/step10d2.py` dùng lại code Step 6 (đặc trưng, lưới trọng số, mô hình LightGBM).

| Họ kiểm định (Holm trong từng họ, đặt trước khi xem kết quả) | So sánh | Số phép |
|---|---|---|
| S0 | QuDAR-confidence − Equal, 8 bộ | 16 |
| S1 | Static-train [cùng bộ] − Equal / − QuDAR-confidence / − QuDAR-llm, 5 bộ có train | 30 |
| S2 | Static-train [các bộ khác] − QuDAR-confidence, 8 bộ (không dùng nhãn của chính bộ đó) | 16 |
| S3 | Model A / Model B [5 bộ train gộp] − Static-train [5 bộ gộp], 8 bộ | 32 |

Thêm phần mô tả (không kiểm định): **đường cong số nhãn**, chọn trọng số tĩnh từ n query train ngẫu nhiên (n = 10 … toàn bộ, 100 lần rút), xem nDCG@10 trên test đạt được và tỉ lệ lần rút vượt QuDAR-confidence.

- **Accelerator:** None (CPU). **Internet:** On
- **Input (10 dataset):** `qudar-runs`, `qudar-runs-2`, `qudar-grid`, `qudar-gridgrid-2`, `qudar-beir-data`, `qudar-beir-data-2`, `qudar-expansions`, `qudar-expansions-2`, `qudar-llm` (gpt-4o-mini, Step 4), `qudar-llm-gpt-new` (gpt-4o-mini, Step 10c-2)
- **Thời gian ước tính:** đọc 13 run (4 run lớn của HotpotQA và FEVER mỗi run 3–6 phút) khoảng 25–40 phút; huấn luyện 2 mô hình LightGBM trên khoảng 6,2 triệu dòng, mỗi mô hình 10–30 phút; kiểm định và đường cong số nhãn vài phút. **Tổng khoảng 1–2 giờ**, có thể lâu hơn.
- **Cách chạy:** Save Version → **Save & Run All (Commit)**.

In [1]:
import glob, os
def found(pattern, keep=lambda p: True):
    return sorted(p for p in glob.glob(pattern, recursive=True) if keep(p))

!git clone -q --depth 1 https://github.com/KarryChuong/CS419-IR-QuDAR-replication.git /kaggle/working/repo
REPO = "/kaggle/working/repo"
assert os.path.exists(f"{REPO}/scripts/step10d2.py"), "main chưa có scripts/step10d2.py: merge PR Step 10d-1 trước"

TEST  = ["fiqa_test", "scidocs_test", "climate-fever_test", "hotpotqa_test",
         "scifact_test", "nfcorpus_test", "dbpedia-entity_test", "fever_test"]
TRAIN = ["fiqa_train", "hotpotqa_train", "scifact_train", "nfcorpus_train", "fever_train"]
problems = []
for n in TEST + TRAIN:
    d, s = n.rsplit("_", 1)
    for what, pat in [("run", f"/kaggle/input/**/{n}/retrieval_result.json"),
                      ("lưới trọng số", f"/kaggle/input/**/{n}/grid_metrics.npz"),
                      ("điểm gpt-4o-mini", f"/kaggle/input/**/llm_scores_paper_{n}.csv"),
                      ("query mở rộng", f"/kaggle/input/**/expanded_queries_{d}_{s}_expansion.json"),
                      ("queries", f"/kaggle/input/**/{d}/queries.jsonl")]:
        if not found(pat):
            problems.append(f"{n}: thiếu {what}")
for n in TRAIN:
    if not found(f"/kaggle/input/**/llm_paper_{n}.jsonl"):
        problems.append(f"{n}: thiếu cache llm_paper_{n}.jsonl (để ước lượng thiên lệch vị trí)")
qwen = found("/kaggle/input/**/llm_scores_*.csv", keep=lambda p: "qwen" in p.lower())
if qwen:
    problems.append(f"đang gắn dataset điểm Qwen ({qwen[0]} ...): gỡ ra, script chỉ dùng gpt-4o-mini")
assert not problems, "Kiểm tra input:\n" + "\n".join(problems)
!pip install -q tabulate
print("Input đủ cho 8 bộ test và 5 bộ train.")

Input đủ cho 8 bộ test và 5 bộ train.


In [2]:
!python {REPO}/scripts/step10d2.py --input /kaggle/input --out /kaggle/working/step10d2

[08:10:01] position bias from train (score at position - mean): {1: 0.5505813349942907, 2: -0.010649157410290933, 3: -0.12781584137859436, 4: -0.412116336205405}
[08:10:01] fiqa_train: loading
[08:10:55] fiqa_train: 5500 queries, 63 features
[08:10:55] hotpotqa_train: loading
[08:12:28] hotpotqa_train: 10000 queries, 63 features
[08:12:28] scifact_train: loading
[08:12:36] scifact_train: 809 queries, 63 features
[08:12:36] nfcorpus_train: loading
[08:12:58] nfcorpus_train: 2590 queries, 63 features
[08:12:58] fever_train: loading
[08:15:14] fever_train: 10000 queries, 63 features
[08:15:14] fiqa_test: loading
[08:15:20] fiqa_test: 648 queries, 63 features
[08:15:20] scidocs_test: loading
[08:15:30] scidocs_test: 1000 queries, 63 features
[08:15:30] climate-fever_test: loading
[08:15:52] climate-fever_test: 1535 queries, 63 features
[08:15:52] hotpotqa_test: loading
[08:16:59] hotpotqa_test: 7405 queries, 63 features
[08:16:59] scifact_test: loading
[08:17:03] scifact_test: 300 queries,

In [3]:
# Bản gọn để gửi lại + đóng gói cho repo
import json, shutil
import pandas as pd
pd.set_option("display.width", 300); pd.set_option("display.max_colwidth", 40)
OUT = "/kaggle/working/step10d2"

print("Trọng số tĩnh được chọn (alpha: sparse 0 -> dense 1; beta1/beta2: query gốc 0 -> mở rộng 1)")
print(pd.read_csv(f"{OUT}/static_points.csv").to_string(index=False))

main = pd.read_csv(f"{OUT}/table_main.csv")
sets = list(dict.fromkeys(main["dataset"]))
for metric in ["nDCG@10", "Recall@10"]:
    pv = main.pivot_table(index="method", columns="dataset", values=metric, sort=False)[sets]
    print(f"\n{metric}\n" + pv.round(4).to_string())

t = pd.read_csv(f"{OUT}/sig_tests.csv")
t["cell"] = t["mean(A-B)"].map("{:+.4f}".format) + t["sig (Holm p<0.05)"].fillna("")
pv = t.pivot_table(index=["family", "A", "B", "metric"], columns="set", values="cell", aggfunc="first", sort=False)
print("\nA - B, dấu * = có ý nghĩa sau Holm trong họ\n" + pv.reindex(columns=sets).fillna("").to_string())

b = pd.read_csv(f"{OUT}/label_budget.csv")
print("\nĐường cong số nhãn (nDCG@10 trên test của trọng số tĩnh chọn từ n query train)")
print(b[["dataset", "train queries", "draws", "nDCG@10 median", "nDCG@10 p05", "nDCG@10 p95",
         "share > QuDAR-confidence (nDCG)", "QuDAR-confidence nDCG@10"]].to_string(index=False))

chk = pd.read_csv(f"{OUT}/check.csv")
print("\nĐối chiếu Equal / QuDAR-confidence với eval_results.csv: max |diff| =", chk["max |diff|"].max())
st = main[main["method"] == "Static-train [in-domain]"].set_index("dataset")["Recall@10"]
print("Đối chiếu với Step 6 (Static-train chọn trên train cùng bộ, Recall@10): FiQA", st.get("FiQA"),
      "(Step 6: 0.3584), HotpotQA", st.get("HotpotQA"), "(Step 6: 0.7041)")
print(json.dumps(json.load(open(f"{OUT}/summary.json"))["families"], indent=1, ensure_ascii=False))

shutil.copytree(OUT, "/kaggle/working/for_repo/results/step10/static8", dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/for_repo_step10d2", "zip", "/kaggle/working/for_repo")
!du -sh {OUT} /kaggle/working/for_repo_step10d2.zip

Trọng số tĩnh được chọn (alpha: sparse 0 -> dense 1; beta1/beta2: query gốc 0 -> mở rộng 1)
              config  alpha  beta1  beta2  w_OS  w_OD  w_ES  w_ED
                 all    0.2    0.5    0.1  0.40  0.18  0.40  0.02
      in-domain:fiqa    0.3    0.4    0.2  0.42  0.24  0.28  0.06
  in-domain:hotpotqa    0.4    0.3    0.6  0.42  0.16  0.18  0.24
   in-domain:scifact    0.3    0.2    0.1  0.56  0.27  0.14  0.03
  in-domain:nfcorpus    0.3    0.4    0.3  0.42  0.21  0.28  0.09
     in-domain:fever    0.2    0.6    0.0  0.32  0.20  0.48  0.00
          other:fiqa    0.2    0.5    0.2  0.40  0.16  0.40  0.04
       other:scidocs    0.2    0.5    0.1  0.40  0.18  0.40  0.02
 other:climate-fever    0.2    0.5    0.1  0.40  0.18  0.40  0.02
      other:hotpotqa    0.2    0.6    0.0  0.32  0.20  0.48  0.00
       other:scifact    0.2    0.5    0.2  0.40  0.16  0.40  0.04
      other:nfcorpus    0.2    0.5    0.2  0.40  0.16  0.40  0.04
other:dbpedia-entity    0.2    0.5    0.1  0.40  0